In [1]:
import re
import pandas as pd
import numpy as np
import plotly.express as px
import sys

from clickhouse_connect import get_client
import os
from dotenv import load_dotenv
import hvplot.pandas

load_dotenv()

True

In [2]:
def get_clickhouse_client():
    return get_client(
        host=os.getenv("CLICKHOUSE_HOST", "10.20.0.243"),
        port=int(os.getenv("CLICKHOUSE_PORT", "8123")),
        username=os.getenv("CLICKHOUSE_USERNAME", ""),
        password=os.getenv("CLICKHOUSE_PASSWORD", ""),
        database="algo",
    )

client = get_clickhouse_client()

In [3]:
date = '2026-07-28'

In [4]:
#### Fetching positions from algo.positions

query_position = f"""
SELECT
  timestamp,
  book_id,
  product_id,
  positions
FROM `algo`.`position`
WHERE
  book_id in (337,129,340,342,339,318,343,309,341,332)
  AND toDate(timestamp) = '{date}'
ORDER BY timestamp ASC
"""
pos_df = client.query_df(query_position)
positions = pd.DataFrame(pos_df.positions.to_list())
positions['timestamp'] = pos_df['timestamp']
positions = positions.fillna(0)
positions = positions.reindex(sorted(positions.columns), axis=1)
positions = positions.groupby(['timestamp']).sum().sort_index()

# # convert monthly outrights → consecutive monthly spreads
# # S_i = cumsum(P)_i  (exact when sum of outrights ≈ 0)
months = list(positions.columns)
pos_spreads = positions.cumsum(axis=1).iloc[:, :-1]
pos_spreads.columns = [f"{a}/{b}" for a, b in zip(months[:-1], months[1:])]
pos_spreads = pos_spreads.sort_index()
pos_spreads.head()

,2026-09-01/2026-10-01,2026-10-01/2026-11-01,2026-11-01/2026-12-01,2026-12-01/2027-01-01,2027-01-01/2027-02-01,2027-02-01/2027-03-01,2027-03-01/2027-04-01,2027-04-01/2027-05-01,2027-05-01/2027-06-01,2027-06-01/2027-07-01,...,2028-06-01/2028-07-01,2028-07-01/2028-08-01,2028-08-01/2028-09-01,2028-09-01/2028-10-01,2028-10-01/2028-11-01,2028-11-01/2028-12-01,2028-12-01/2029-02-01,2029-02-01/2029-03-01,2029-03-01/2029-04-01,2029-04-01/2029-06-01
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-07-28 00:01:05.944000+01:00,0.0,-19.0,1.0,8.0,1.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,0.0,-17.0,-15.0,-15.0
2026-07-28 00:01:05.945000+01:00,0.0,-18.0,0.0,9.0,0.0,0.0,1.0,0.0,0.0,2.0,...,7.0,0.0,-8.0,3.0,0.0,-13.0,0.0,-16.0,-15.0,-15.0
2026-07-28 02:32:45.108000+01:00,0.0,-18.0,0.0,9.0,0.0,0.0,1.0,0.0,0.0,2.0,...,7.0,0.0,-8.0,3.0,0.0,-13.0,0.0,-16.0,-15.0,-15.0
2026-07-28 02:32:45.119000+01:00,0.0,-19.0,1.0,8.0,1.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,0.0,-17.0,-15.0,-15.0
2026-07-28 07:36:57.634000+01:00,0.0,-21.0,1.0,8.0,1.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,0.0,-17.0,-15.0,-15.0


In [5]:
#### Nexus Trades → consecutive monthly spread positions

def convert_string_dates_to_actual_val(x):
    mapping = {'Jan':'01', 'Feb':'02', 'Mar':'03', 'Apr':'04', 'May':'05', 'Jun':'06', 'Jul':'07', 'Aug':'08', 'Sep':'09', 'Oct':'10', 'Nov':'11', 'Dec':'12'}

    start = x[0]
    end = x[1]
    start = '20' + start[-2:] + '-' + mapping[start[:3]] + '-01'
    end = '20' + end[-2:] + '-' + mapping[end[:3]] + '-01'
    return start + '/' + end

query_trades = f"SELECT * FROM algo.nexus_trades WHERE toDate(transaction_timestamp) = '{date}' ORDER BY transaction_timestamp ASC"
trades = client.query_df(query_trades)
trades['contract'] = trades.instrument_key.str.split(' ').apply(lambda x: x[2])
trades['contract'] = trades['contract'].str.split('-').apply(lambda x: convert_string_dates_to_actual_val(x))
trades['position'] = np.where(trades['side'].str.lower() == 'buy', trades.quantity, -1.0*trades.quantity)
trades[['start_tenor', 'end_tenor']] = trades['contract'].str.split('/', expand=True)

def to_monthly_spreads(row):
    months = pd.date_range(row['start_tenor'], row['end_tenor'], freq='MS')
    if len(months) < 2:
        return []
    return [
        {
            'transaction_timestamp': row['transaction_timestamp'],
            'contract': f"{m.date()}/{n.date()}",
            'position': row['position'],
        }
        for m, n in zip(months[:-1], months[1:])
    ]

monthly_rows = [r for _, row in trades.iterrows() for r in to_monthly_spreads(row)]
monthly = pd.DataFrame(monthly_rows)

pos_hedged = monthly.groupby(['transaction_timestamp', 'contract'])['position'].sum().reset_index()
pos_hedged = (
    pos_hedged.pivot(index='transaction_timestamp', columns='contract', values='position')
    .fillna(0)
    .sort_index()
)
pos_hedged = pos_hedged.reindex(sorted(pos_hedged.columns), axis=1)
pos_hedged_cumsum = pos_hedged.cumsum()


In [6]:
#### Nexus Transfers

query = f"SELECT * FROM algo.nexus_transfers WHERE toDate(timestamp) = '{date}' ORDER BY timestamp ASC"
df = client.query_df(query)
df['position'] = np.where(df['side'].str.lower() == 'sell', df.quantity, -1.0 * df.quantity)

def to_monthly_spreads(row):
    months = pd.date_range(row['start_tenor'], row['end_tenor'], freq='MS')
    if len(months) < 2:
        return []
    return [
        {
            'timestamp': row['timestamp'],
            'contract': f"{m.date()}/{n.date()}",
            'position': row['position'],
        }
        for m, n in zip(months[:-1], months[1:])
    ]

monthly_rows = [r for _, row in df.iterrows() for r in to_monthly_spreads(row)]
monthly = pd.DataFrame(monthly_rows)

pos_summary = monthly.groupby(['timestamp', 'contract'])['position'].sum().reset_index()
pos_transfer = (
    pos_summary.pivot(index='timestamp', columns='contract', values='position')
    .fillna(0)
    .sort_index()
)
pos_transfer = pos_transfer.reindex(sorted(pos_transfer.columns), axis=1)

t0 = pos_transfer.index[0]
prior = pos_spreads.loc[:t0].iloc[-1]

all_contracts = pos_transfer.columns.union(prior.index)
pos_transfer = pos_transfer.reindex(columns=all_contracts).fillna(0)

pos_transfer = pos_transfer.reindex(
    index=pos_spreads.index,
    columns=pos_spreads.columns,
).fillna(0)
pos_transfer.iloc[0, :] = pos_transfer.iloc[0, :] + prior.reindex(all_contracts).fillna(0)
pos_transfer_cumsum = pos_transfer.cumsum()

ValueError: Must have equal len keys and value when setting with an iterable

### Aligning trades and transfers with algo.positions

In [15]:
transfer_aligned = pos_transfer_cumsum.reindex(
    index=pos_spreads.index,
    columns=pos_spreads.columns,
    method='ffill',  # or 'ffill' for last-known position
)

hedges_aligned = pos_hedged_cumsum.reindex(
    index=pos_spreads.index,
    columns=pos_spreads.columns,
    method='ffill',  # or 'ffill' for last-known position
).fillna(0)

inventory_remaining = transfer_aligned + hedges_aligned

In [14]:
tenor = '2027-02-01/2027-03-01'
inventory_remaining[tenor]

timestamp
2026-07-28 00:01:05.944000+01:00     NaN
2026-07-28 00:01:05.945000+01:00     NaN
2026-07-28 02:32:45.108000+01:00     NaN
2026-07-28 02:32:45.119000+01:00     NaN
2026-07-28 07:36:57.634000+01:00     NaN
                                    ... 
2026-07-28 21:00:09.034000+01:00    65.0
2026-07-28 21:00:09.084000+01:00    65.0
2026-07-28 21:00:09.085000+01:00    65.0
2026-07-28 22:32:45.264000+01:00    65.0
2026-07-28 22:32:45.462000+01:00    65.0
Name: 2027-02-01/2027-03-01, Length: 4769, dtype: float64

In [8]:
pos_hedged_cumsum

contract,2026-09-01/2026-10-01,2026-10-01/2026-11-01,2026-11-01/2026-12-01,2026-12-01/2027-01-01,2027-01-01/2027-02-01,2027-02-01/2027-03-01,2027-03-01/2027-04-01,2027-04-01/2027-05-01,2027-05-01/2027-06-01,2027-06-01/2027-07-01,...,2028-06-01/2028-07-01,2028-07-01/2028-08-01,2028-08-01/2028-09-01,2028-09-01/2028-10-01,2028-10-01/2028-11-01,2028-11-01/2028-12-01,2029-02-01/2029-03-01,2029-03-01/2029-04-01,2029-04-01/2029-05-01,2029-05-01/2029-06-01
transaction_timestamp,,,,,,,,,,,,,,,,,,,,,
2026-07-28 07:36:59.358000+01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
2026-07-28 07:37:00.264000+01:00,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
2026-07-28 07:37:00.266000+01:00,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
2026-07-28 07:37:00.361000+01:00,0.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
2026-07-28 07:37:01.434000+01:00,0.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-1.0,0.0,0.0,2.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-28 19:41:26.245000+01:00,1.0,1820.0,341.0,667.0,779.0,634.0,440.0,136.0,-46.0,-145.0,...,-15.0,-8.0,15.0,-7.0,-1.0,25.0,22.0,29.0,29.0,29.0
2026-07-28 19:42:57.957000+01:00,1.0,1820.0,341.0,673.0,783.0,634.0,440.0,136.0,-46.0,-145.0,...,-15.0,-8.0,15.0,-7.0,-1.0,25.0,22.0,29.0,29.0,29.0
2026-07-28 19:43:40.240000+01:00,1.0,1823.0,344.0,673.0,783.0,634.0,440.0,136.0,-46.0,-145.0,...,-15.0,-8.0,15.0,-7.0,-1.0,25.0,22.0,29.0,29.0,29.0


In [7]:
transfer_aligned

contract,2026-09-01/2026-10-01,2026-10-01/2026-11-01,2026-11-01/2026-12-01,2026-12-01/2027-01-01,2027-01-01/2027-02-01,2027-02-01/2027-03-01,2027-03-01/2027-04-01,2027-04-01/2027-05-01,2027-05-01/2027-06-01,2027-06-01/2027-07-01,...,2028-06-01/2028-07-01,2028-07-01/2028-08-01,2028-08-01/2028-09-01,2028-09-01/2028-10-01,2028-10-01/2028-11-01,2028-11-01/2028-12-01,2029-02-01/2029-03-01,2029-03-01/2029-04-01,2029-04-01/2029-05-01,2029-05-01/2029-06-01
transaction_timestamp,,,,,,,,,,,,,,,,,,,,,
2026-07-28 07:36:59.358000+01:00,0.0,-22.0,-12.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0
2026-07-28 07:37:00.264000+01:00,0.0,-22.0,-12.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0
2026-07-28 07:37:00.266000+01:00,0.0,-22.0,-12.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0
2026-07-28 07:37:00.361000+01:00,0.0,-22.0,-12.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0
2026-07-28 07:37:01.434000+01:00,0.0,-22.0,-12.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-28 19:41:26.245000+01:00,-1.0,-1831.0,-341.0,-710.0,-786.0,-569.0,-412.0,-138.0,-6.0,133.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0
2026-07-28 19:42:57.957000+01:00,-1.0,-1831.0,-341.0,-710.0,-786.0,-569.0,-412.0,-138.0,-6.0,133.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0
2026-07-28 19:43:40.240000+01:00,-1.0,-1831.0,-341.0,-710.0,-786.0,-569.0,-412.0,-138.0,-6.0,133.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-17.0,-15.0,0.0,0.0


In [10]:
pos_remaining = (transfer_aligned + pos_hedged_cumsum)
last_idx = pos_remaining.index[-1]
last_idx

Timestamp('2026-07-28 19:52:35.416000+0100', tz='Europe/London')

In [12]:
pos_remaining

contract,2026-09-01/2026-10-01,2026-10-01/2026-11-01,2026-11-01/2026-12-01,2026-12-01/2027-01-01,2027-01-01/2027-02-01,2027-02-01/2027-03-01,2027-03-01/2027-04-01,2027-04-01/2027-05-01,2027-05-01/2027-06-01,2027-06-01/2027-07-01,...,2028-06-01/2028-07-01,2028-07-01/2028-08-01,2028-08-01/2028-09-01,2028-09-01/2028-10-01,2028-10-01/2028-11-01,2028-11-01/2028-12-01,2029-02-01/2029-03-01,2029-03-01/2029-04-01,2029-04-01/2029-05-01,2029-05-01/2029-06-01
transaction_timestamp,,,,,,,,,,,,,,,,,,,,,
2026-07-28 07:36:59.358000+01:00,0.0,-22.0,-12.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-15.0,-15.0,0.0,0.0
2026-07-28 07:37:00.264000+01:00,0.0,-19.0,-9.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-15.0,-15.0,0.0,0.0
2026-07-28 07:37:00.266000+01:00,0.0,-18.0,-8.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-15.0,-15.0,0.0,0.0
2026-07-28 07:37:00.361000+01:00,0.0,-17.0,-7.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,4.0,0.0,-13.0,-15.0,-15.0,0.0,0.0
2026-07-28 07:37:01.434000+01:00,0.0,-17.0,-7.0,-5.0,0.0,0.0,1.0,1.0,1.0,2.0,...,7.0,0.0,-8.0,3.0,0.0,-13.0,-15.0,-15.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-28 19:41:26.245000+01:00,0.0,-11.0,0.0,-43.0,-7.0,65.0,28.0,-2.0,-52.0,-12.0,...,-8.0,-8.0,7.0,-3.0,-1.0,12.0,5.0,14.0,29.0,29.0
2026-07-28 19:42:57.957000+01:00,0.0,-11.0,0.0,-37.0,-3.0,65.0,28.0,-2.0,-52.0,-12.0,...,-8.0,-8.0,7.0,-3.0,-1.0,12.0,5.0,14.0,29.0,29.0
2026-07-28 19:43:40.240000+01:00,0.0,-8.0,3.0,-37.0,-3.0,65.0,28.0,-2.0,-52.0,-12.0,...,-8.0,-8.0,7.0,-3.0,-1.0,12.0,5.0,14.0,29.0,29.0


In [11]:
pos_spreads[pos_spreads.index>=last_idx]

,2026-09-01/2026-10-01,2026-10-01/2026-11-01,2026-11-01/2026-12-01,2026-12-01/2027-01-01,2027-01-01/2027-02-01,2027-02-01/2027-03-01,2027-03-01/2027-04-01,2027-04-01/2027-05-01,2027-05-01/2027-06-01,2027-06-01/2027-07-01,...,2028-06-01/2028-07-01,2028-07-01/2028-08-01,2028-08-01/2028-09-01,2028-09-01/2028-10-01,2028-10-01/2028-11-01,2028-11-01/2028-12-01,2028-12-01/2029-02-01,2029-02-01/2029-03-01,2029-03-01/2029-04-01,2029-04-01/2029-06-01
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-07-28 19:52:35.639000+01:00,0.0,-13.0,1.0,-14.0,0.0,32.0,17.0,-5.0,-28.0,-2.0,...,0.0,-7.0,0.0,1.0,0.0,1.0,1.0,-10.0,0.0,0.0
2026-07-28 19:52:35.691000+01:00,0.0,-13.0,1.0,-14.0,0.0,32.0,17.0,-5.0,-28.0,-1.0,...,-1.0,-8.0,-1.0,0.0,-1.0,0.0,0.0,-11.0,-1.0,-1.0
2026-07-28 21:00:08.932000+01:00,0.0,-13.0,1.0,-14.0,0.0,32.0,17.0,-5.0,-28.0,-1.0,...,-1.0,-8.0,-1.0,0.0,-1.0,0.0,0.0,-11.0,-1.0,-1.0
2026-07-28 21:00:08.933000+01:00,0.0,-13.0,2.0,-12.0,-3.0,33.0,12.0,3.0,-24.0,-7.0,...,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0
2026-07-28 21:00:08.984000+01:00,0.0,-13.0,1.0,-13.0,0.0,32.0,17.0,-5.0,-28.0,-1.0,...,-1.0,-8.0,-1.0,0.0,-1.0,0.0,0.0,-11.0,-1.0,-1.0
2026-07-28 21:00:08.985000+01:00,0.0,-13.0,2.0,-13.0,-3.0,33.0,12.0,3.0,-24.0,-7.0,...,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0
2026-07-28 21:00:09.034000+01:00,0.0,-13.0,1.0,-13.0,-2.0,32.0,14.0,-1.0,-26.0,-4.0,...,-1.0,-4.0,-1.0,0.0,-1.0,0.0,0.0,-6.0,0.0,0.0
2026-07-28 21:00:09.034000+01:00,0.0,-13.0,2.0,-13.0,-1.0,33.0,15.0,-1.0,-26.0,-4.0,...,0.0,-4.0,0.0,0.0,0.0,-1.0,0.0,-5.0,-1.0,-1.0
2026-07-28 21:00:09.084000+01:00,0.0,-13.0,1.0,-13.0,-2.0,32.0,14.0,-1.0,-26.0,-4.0,...,-1.0,-4.0,-1.0,0.0,-1.0,0.0,0.0,-6.0,-1.0,0.0
